# Semi-Automatic Correct / Incorrect Prediction Filter

This notebook implements two semi-automatic approaches for judging whether SAM3-predicted masks are accurate, applied to two different dataset types:

- **Crack images** — the original dataset only has YOLO bounding-box labels (no GT masks), so filtering is done by comparing predicted masks against those boxes using IoU or geometric criteria.
- **Pin images** — the dataset has ground-truth binary masks, so filtering is done by comparing the number of detected pin components between GT and prediction.

The notebook is split into three main parts:

1. **Crack — IoU-based filtering** (Method 1): converts YOLO boxes to binary GT masks and thresholds on IoU.
2. **Crack — Multi-criteria filtering** (Method 2): uses fill ratio, outside ratio, and component count to give a more nuanced quality judgement.
3. **Pin — GT mask comparison**: counts connected components (individual pins) in the GT and predicted masks and flags mismatches.

---
## Part 1 — Cracks (YOLO labels only)

The crack dataset has no GT masks — only YOLO bounding-box `.txt` files. The two methods below both work around this limitation by deriving a proxy GT from those boxes.

### Method 1: IoU-based Filtering

Each YOLO bounding box is rasterised into a binary GT mask (a filled rectangle). The Intersection-over-Union (IoU) between the predicted mask and this GT rectangle is then computed. Predictions that meet the IoU threshold (empirically tuned to `0.05`) are copied to `correct_masks` / `correct_images`; those that fall below go to `incorrect_masks` / `incorrect_images`.

This is a quick first-pass filter. Because the GT is just a filled box rather than a true mask, the threshold intentionally has to be kept very low.

In [ ]:
import shutil
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import json
from scipy import ndimage

In [ ]:
mask_dir = Path("/home/hpc_storage/data/projects/userA/sam3_data/war_crack/syn_data/masks")
img_dir = Path("/home/hpc_storage/data/projects/userA/sam3_data/war_crack/syn_data/images")
label_dir = Path("/home/hpc_storage/data/projects/userA/sam3_data/war_crack/syn_data/labels")

In [ ]:
label_dir.exists()

In [ ]:
def yolo_labels_to_mask(label_path: Path, img_w: int, img_h: int) -> np.ndarray:
    """Convert YOLO bounding-box labels to a binary GT mask (0/255)."""
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    if not label_path.exists():
        return mask
    with open(label_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            _, xc, yc, bw, bh = map(float, parts[:5])
            x1 = int((xc - bw / 2) * img_w)
            y1 = int((yc - bh / 2) * img_h)
            x2 = int((xc + bw / 2) * img_w)
            y2 = int((yc + bh / 2) * img_h)
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img_w, x2), min(img_h, y2)
            mask[y1:y2, x1:x2] = 255
    return mask


def mask_iou(pred_mask: np.ndarray, gt_mask: np.ndarray) -> float:
    """Compute IoU between two binary masks (values 0 or 255)."""
    pred_bin = pred_mask > 0
    gt_bin   = gt_mask   > 0
    intersection = np.logical_and(pred_bin, gt_bin).sum()
    union        = np.logical_or(pred_bin, gt_bin).sum()
    if union == 0:
        # Both masks empty — treat as correct (no object, no prediction)
        return 1.0
    return float(intersection) / float(union)


In [ ]:
iou_threshold = 0.05   #tried trial and error for this

correct_dir_images   = mask_dir.parent / "correct_images"
correct_dir_masks    = mask_dir.parent / "correct_masks"
incorrect_dir_images = mask_dir.parent / "incorrect_images"
incorrect_dir_masks  = mask_dir.parent / "incorrect_masks"
correct_dir_images.mkdir(parents=True, exist_ok=True)
correct_dir_masks.mkdir(parents=True, exist_ok=True)
incorrect_dir_images.mkdir(parents=True, exist_ok=True)
incorrect_dir_masks.mkdir(parents=True, exist_ok=True)

correct_count = 0
incorrect_count = 0
skipped_count = 0

for mask_path in tqdm(sorted(mask_dir.glob("*.png"))):
    stem = mask_path.stem  

    # Find original image (support .png / .jpg / .jpeg)
    img_path = None
    for ext in (".png", ".jpg", ".jpeg"):
        candidate = img_dir / f"{stem}{ext}"
        if candidate.exists():
            img_path = candidate
            break

    if img_path is None:
        print(f"[SKIP] No source image found for: {stem}")
        skipped_count += 1
        continue

    label_path = label_dir / f"{stem}.txt"

    # Load predicted mask
    pred_mask = np.array(Image.open(mask_path).convert("L"))
    img_h, img_w = pred_mask.shape

    # Build GT mask from YOLO labels
    gt_mask = yolo_labels_to_mask(label_path, img_w, img_h)

    iou = mask_iou(pred_mask, gt_mask)

    if iou >= iou_threshold:
        shutil.copy2(mask_path, correct_dir_masks / mask_path.name)
        shutil.copy2(img_path, correct_dir_images / img_path.name)
        correct_count += 1
    else:
        shutil.copy2(mask_path, incorrect_dir_masks / mask_path.name)
        shutil.copy2(img_path, incorrect_dir_images / img_path.name)
        incorrect_count += 1

print(f"Done.")
print(f"  Correct   (IoU >= {iou_threshold}): {correct_count}")
print(f"  Incorrect (IoU <  {iou_threshold}): {incorrect_count}")
if skipped_count:
    print(f"  Skipped (no source image): {skipped_count}")


### Method 2: Multi-Criteria Quality Filtering

Method 1 turned out to be noisy because a low IoU threshold catches many false positives. This second approach replaces a single IoU score with three interpretable geometric checks applied to the crack dataset:

| Criterion | What it measures | Threshold used |
|---|---|---|
| `fill_ratio` | What fraction of the YOLO bbox area the predicted mask covers | `0.001 – 0.60` |
| `outside_ratio` | What fraction of mask pixels fall *outside* the bbox | `≤ 0.30` |
| `n_components` | Number of disconnected blobs in the predicted mask | `≤ 1` |

A prediction is marked **correct** only if it passes *all* three checks. Failures on any criterion are recorded and the image/mask/overlay triplet is copied to the appropriate output folder. Once again the criterion values were decided upon by trial and error

In [ ]:
import shutil
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from scipy import ndimage

In [ ]:
IMG_DIR     = Path("/home/hpc_storage/data/projects/userA/sam3_data/jubna_images/test/patches/images")
LABEL_DIR   = Path("/home/hpc_storage/data/projects/userA/sam3_data/jubna_images/test/patches/labels")
MASK_DIR    = Path("/home/hpc_storage/data/projects/userA/sam3_data/jubna_images/test/masks_and_overlays/masks")
OVERLAY_DIR = Path("/home/hpc_storage/data/projects/userA/sam3_data/jubna_images/test/masks_and_overlays/overlays")
output_dir  = Path("/home/hpc_storage/data/projects/userA/sam3_data/jubna_images/test/masks_and_overlays/filter_output")
OUT_CORRECT   = output_dir / "correct"
OUT_INCORRECT = output_dir / "incorrect"

# Thresholds
MIN_FILL_RATIO     = 0.001   # mask must cover at least 0.1% of bbox
MAX_FILL_RATIO     = 0.60    # mask covers no more than 60% of bbox
MAX_OUTSIDE_RATIO  = 0.30    # at most 30% of mask pixels outside bbox
MAX_COMPONENTS     = 1       # no more than 1 connected blobs


OUT_CORRECT.mkdir(parents=True, exist_ok=True)
OUT_INCORRECT.mkdir(parents=True, exist_ok=True)

IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tiff")

In [ ]:
def parse_yolo_bboxes(label_path: Path, img_w: int, img_h: int):
    """Return list of (x1, y1, x2, y2) pixel bboxes from YOLO label file."""
    bboxes = []
    if not label_path.exists():
        return bboxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            _, xc, yc, bw, bh = map(float, parts[:5])
            x1 = max(0, int((xc - bw / 2) * img_w))
            y1 = max(0, int((yc - bh / 2) * img_h))
            x2 = min(img_w, int((xc + bw / 2) * img_w))
            y2 = min(img_h, int((yc + bh / 2) * img_h))
            bboxes.append((x1, y1, x2, y2))
    return bboxes

In [ ]:
def evaluate_mask(pred_mask: np.ndarray, bboxes: list) -> dict:
    """
    Returns a dict with quality metrics:
      is_correct, fill_ratio, outside_ratio, n_components, reason
    """
    pred_bin = pred_mask > 0
    mask_pixels = pred_bin.sum()
    has_label = len(bboxes) > 0
    has_mask  = mask_pixels > 0

    # Case 1: no label and no mask → correct (true negative)
    if not has_label and not has_mask:
        return {"is_correct": True, "reason": "true_negative"}

    # Case 2: no label but model predicted something → incorrect
    if not has_label and has_mask:
        return {"is_correct": False, "reason": "false_positive_no_label"}

    # Case 3: has label but no mask → incorrect
    if has_label and not has_mask:
        return {"is_correct": False, "reason": "empty_mask"}

    # Case 4: has label and has mask → run quality checks
    h, w = pred_mask.shape
    bbox_mask = np.zeros((h, w), dtype=bool)
    total_bbox_area = 0
    for (x1, y1, x2, y2) in bboxes:
        bbox_mask[y1:y2, x1:x2] = True
        total_bbox_area += (x2 - x1) * (y2 - y1)

    if total_bbox_area == 0:
        return {"is_correct": False, "reason": "zero_area_bbox"}

    inside = np.logical_and(pred_bin, bbox_mask).sum()
    fill_ratio = inside / total_bbox_area

    outside = mask_pixels - inside
    outside_ratio = outside / mask_pixels

    labelled, n_components = ndimage.label(pred_bin)

    reasons = []
    if fill_ratio < MIN_FILL_RATIO:
        reasons.append(f"fill_too_low({fill_ratio:.4f})")
    if fill_ratio > MAX_FILL_RATIO:
        reasons.append(f"fill_too_high({fill_ratio:.4f})")
    if outside_ratio > MAX_OUTSIDE_RATIO:
        reasons.append(f"outside_too_high({outside_ratio:.4f})")
    if n_components > MAX_COMPONENTS:
        reasons.append(f"too_many_components({n_components})")

    is_correct = len(reasons) == 0

    return {
        "is_correct":     is_correct,
        "fill_ratio":     round(fill_ratio, 4),
        "outside_ratio":  round(outside_ratio, 4),
        "n_components":   n_components,
        "reason":         "pass" if is_correct else " | ".join(reasons),
    }

In [ ]:
# Main loop 
correct = incorrect = skipped = 0

for img_path in tqdm(sorted(IMG_DIR.iterdir()), desc="Evaluating"):
    if img_path.suffix.lower() not in IMG_EXTS:
        continue
    stem = img_path.stem

    # Find matching mask
    mask_path = MASK_DIR / f"{stem}_mask.png"
    if not mask_path.exists():
        mask_path = MASK_DIR / f"{stem}.png"

    # Find matching overlay
    overlay_path = OVERLAY_DIR / f"{stem}_overlay.png"

    label_path = LABEL_DIR / f"{stem}.txt"

    img = Image.open(img_path).convert("RGB")
    img_w, img_h = img.size

    if mask_path.exists():
        pred_mask = np.array(Image.open(mask_path).convert("L"))
    else:
        pred_mask = np.zeros((img_h, img_w), dtype=np.uint8)

    bboxes = parse_yolo_bboxes(label_path, img_w, img_h)
    result = evaluate_mask(pred_mask, bboxes)

    dest = OUT_CORRECT if result["is_correct"] else OUT_INCORRECT

    shutil.copy2(img_path, dest / img_path.name)
    if mask_path.exists():
        shutil.copy2(mask_path, dest / mask_path.name)
    if overlay_path.exists():
        shutil.copy2(overlay_path, dest / overlay_path.name)

    if result["is_correct"]:
        correct += 1
    else:
        incorrect += 1

print(f"\nDone.")
print(f"  Correct:   {correct}")
print(f"  Incorrect: {incorrect}")
print(f"  Skipped:   {skipped}")


### Section 3: Output Folder Audit

This was used for debugging. After copying files to `correct/` or `incorrect/`, this cell verifies that every stem has all three expected file types — the original image, a `_mask.png`, and an `_overlay.png`. Any stems with missing files are printed so naming mismatches or incomplete copies can be caught early.

In [ ]:

from pathlib import Path
from collections import defaultdict

# ── Point this at any output folder you want to audit ──────────────────────────
AUDIT_DIR = OUT_CORRECT   # or OUT_INCORRECT
IMG_EXTS  = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}
# ──────────────────────────────────────────────────────────────────────────────

files = list(AUDIT_DIR.iterdir())

images   = {}   # stem -> path
masks    = {}   # stem -> path  (filename ends with _mask)
overlays = {}   # stem -> path  (filename ends with _overlay)

for f in files:
    name = f.stem
    if name.endswith("_mask"):
        masks[name[:-5]] = f          # strip _mask to get base stem
    elif name.endswith("_overlay"):
        overlays[name[:-8]] = f       # strip _overlay to get base stem
    elif f.suffix.lower() in IMG_EXTS:
        images[name] = f

all_stems = set(images) | set(masks) | set(overlays)

missing_mask    = []
missing_overlay = []
missing_image   = []

for stem in sorted(all_stems):
    if stem not in images:
        missing_image.append(stem)
    if stem not in masks:
        missing_mask.append(stem)
    if stem not in overlays:
        missing_overlay.append(stem)

print(f"Audit folder : {AUDIT_DIR}")
print(f"Total unique stems : {len(all_stems)}")
print(f"  Images   : {len(images)}")
print(f"  Masks    : {len(masks)}")
print(f"  Overlays : {len(overlays)}")
print()

if missing_image:
    print(f"Missing IMAGE  ({len(missing_image)}):")
    for s in missing_image:
        print(f"  {s}")
else:
    print("No missing images.")

if missing_mask:
    print(f"\nMissing MASK   ({len(missing_mask)}):")
    for s in missing_mask:
        print(f"  {s}")
else:
    print("No missing masks.")

if missing_overlay:
    print(f"\nMissing OVERLAY ({len(missing_overlay)}):")
    for s in missing_overlay:
        print(f"  {s}")
else:
    print("No missing overlays.")


---
## Part 2 — Pins (ground-truth masks available)

The pin dataset (`multi_test/et_pin`) has proper binary GT masks, so the predicted mask is compared directly to the GT mask using **connected-component (pin) counting**.

The logic is straightforward — if the predicted mask contains the same number of pin blobs as the GT mask, the prediction is considered correct. Extra blobs indicate a false positive (the model hallucinated a pin); fewer blobs indicate a missed detection.

A `min_gt_pin_pixels` threshold (default `120 px`(found through trial and error)) is applied to the GT to discard edge-clipped.

In [ ]:
# Load pre-computed IoU results from JSON (keyed by filename stem)
def load_json_iou(json_path: Path) -> dict:
    """Returns dict: stem -> IoU result dict."""
    iou_lookup: dict = {}
    if json_path.exists():
        with open(json_path) as f:
            json_data = json.load(f)
        for r in json_data.get("per_image_results", []):
            stem = Path(r["file_name"]).stem
            iou_lookup[stem] = r
        print(f"Loaded {len(iou_lookup)} entries from JSON.")
    else:
        print("JSON not found — IoU will be computed from scratch.")
    return iou_lookup

In [ ]:

def count_pins(mask: np.ndarray, min_pixels: int = 0) -> int:
    """Return number of connected components (pins) in a binary mask.
    Components with fewer than min_pixels pixels are ignored."""
    labelled, n = ndimage.label(mask > 127)
    if min_pixels <= 0 or n == 0:
        return n
    sizes = ndimage.sum(mask > 127, labelled, range(1, n + 1))
    return int(np.sum(np.array(sizes) >= min_pixels))


In [ ]:
def compute_iou(gt: np.ndarray, pred: np.ndarray) -> float:
    gt_bin   = gt   > 0
    pred_bin = pred > 0
    inter = np.logical_and(gt_bin, pred_bin).sum()
    union = np.logical_or(gt_bin, pred_bin).sum()
    return float(inter) / float(union) if union > 0 else 1.0

In [ ]:

def main_comparison(pred_mask_dir: Path, gt_mask_dir: Path, iou_lookup: dict, iou_threshold: float, min_gt_pin_pixels: int = 0):
    incorrect = []
    skipped   = []
    correct_count = 0

    gt_files = sorted(p for p in gt_mask_dir.iterdir() if p.suffix.lower() in IMG_EXTS)

    for gt_path in tqdm(gt_files, desc="Filtering"):
        stem = gt_path.stem

        # Find predicted mask — try <stem>_mask.png first, then <stem>.png
        pred_path = pred_mask_dir / f"{stem}_mask.png"
        if not pred_path.exists():
            pred_path = pred_mask_dir / f"{stem}.png"
        if not pred_path.exists():
            skipped.append(stem)
            continue

        gt_mask   = np.array(Image.open(gt_path).convert("L"))
        pred_mask = np.array(Image.open(pred_path).convert("L"))

        # Pin count check 
        # Ignore GT pins smaller than min_gt_pin_pixels (edge-clipped artefacts)
        gt_pins   = count_pins(gt_mask, min_pixels=min_gt_pin_pixels)
        pred_pins = count_pins(pred_mask)
        pin_count_ok = (gt_pins == pred_pins)

        # Classify
        if pin_count_ok:
            correct_count += 1
        else:
            diff = pred_pins - gt_pins
            kind = "extra" if diff > 0 else "missing"
            reason = f"pin_count gt={gt_pins} pred={pred_pins} ({kind})"
            incorrect.append({"name": stem, "reason": reason, "gt_path": gt_path})

    #Results
    total_processed = len(gt_files) - len(skipped)
    print(f"GT masks found      : {len(gt_files)}")
    print(f"Skipped (no pred)   : {len(skipped)}")
    print(f"Processed           : {total_processed}")
    print(f"  Correct           : {correct_count}")
    print(f"  Incorrect         : {len(incorrect)}")
    if min_gt_pin_pixels:
        print(f"  (GT pins < {min_gt_pin_pixels}px ignored)")

    if incorrect:
        print(f"\nIncorrect predictions ({len(incorrect)}):")
        for item in incorrect:
            print(f"  {item['name']}  [{item['reason']}]")

    return incorrect


In [ ]:
# Optional: For each incorrect item, copy the original image, GT mask, predicted mask, and overlay (if exists) into a separate folder for manual review.
def copy_incorrect_files(
    incorrect: list,
    src_img_dir: Path,
    src_mask_dir: Path,
    dest_dir: Path,
    src_overlay_dir: Path = None,
):
    """Save concatenated image [original | GT mask | pred mask (| overlay)] for each incorrect item."""
    dest_dir.mkdir(parents=True, exist_ok=True)

    for item in incorrect:
        stem       = item["name"]
        reason     = item["reason"]
        gt_path    = item.get("gt_path")

        # Find source image
        src_img_path = None
        for ext in IMG_EXTS:
            candidate = src_img_dir / f"{stem}{ext}"
            if candidate.exists():
                src_img_path = candidate
                break
        if src_img_path is None:
            print(f"[WARN] No source image found for: {stem}")
            continue

        # Find predicted mask
        src_mask_path = None
        for suffix in ["_mask", ""]:
            for ext in IMG_EXTS:
                candidate = src_mask_dir / f"{stem}{suffix}{ext}"
                if candidate.exists():
                    src_mask_path = candidate
                    break
            if src_mask_path:
                break
        if src_mask_path is None:
            print(f"[WARN] No predicted mask found for: {stem}")
            continue

        # Find overlay
        src_overlay_path = None
        if src_overlay_dir is not None:
            for suffix in ["_overlay", ""]:
                for ext in IMG_EXTS:
                    candidate = src_overlay_dir / f"{stem}{suffix}{ext}"
                    if candidate.exists():
                        src_overlay_path = candidate
                        break
                if src_overlay_path:
                    break

        # Build panel list: image | GT mask | pred mask [| overlay]
        panels = [Image.open(src_img_path).convert("RGB")]
        if gt_path and Path(gt_path).exists():
            panels.append(Image.open(gt_path).convert("RGB"))
        panels.append(Image.open(src_mask_path).convert("RGB"))
        if src_overlay_path:
            panels.append(Image.open(src_overlay_path).convert("RGB"))

        # Resize all panels to the same height (tallest wins)
        target_h = max(p.height for p in panels)
        resized = []
        for p in panels:
            if p.height != target_h:
                scale = target_h / p.height
                p = p.resize((int(p.width * scale), target_h), Image.LANCZOS)
            resized.append(p)

        # Concatenate horizontally
        total_w  = sum(p.width for p in resized)
        combined = Image.new("RGB", (total_w, target_h))
        x_offset = 0
        for p in resized:
            combined.paste(p, (x_offset, 0))
            x_offset += p.width

        # Save
        safe_reason = reason.replace(" ", "_").replace("|", "-").replace("/", "-")
        out_path    = dest_dir / f"{stem}__{safe_reason}.png"
        combined.save(out_path)


In [ ]:
GT_MASK_DIR   = Path("/home/hpc_storage/data/projects/userA/sam3_data/multi_test/masks/et_pin")
PRED_MASK_DIR = Path("/home/hpc_storage/data/projects/userA/sam3_data/experiments/testing/masks_and_overlays/masks")
JSON_PATH     = Path("/home/hpc_storage/data/projects/userA/sam3_data/experiments/testing/evaluation_output/foreground_iou_summary.json")

IOU_THRESHOLD      = 0.50
MIN_GT_PIN_PIXELS  = 120   # GT components smaller than this are ignored (edge-clipped pins)
IMG_EXTS           = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}
iou_lookup         = load_json_iou(JSON_PATH)
incorrect_1          = main_comparison(PRED_MASK_DIR, GT_MASK_DIR, iou_lookup, IOU_THRESHOLD, min_gt_pin_pixels=MIN_GT_PIN_PIXELS)


In [ ]:
dest_incorrect     = Path("/home/hpc_storage/data/projects/userA/sam3_data/experiments/testing/incorrect_visuals")
source_overlay_dir = Path("/home/hpc_storage/data/projects/userA/sam3_data/experiments/testing/masks_and_overlays/overlays")
src_img_dir        = Path("/home/hpc_storage/data/projects/userA/sam3_data/multi_test/images/et_pin")

copy_incorrect_files(
    incorrect=incorrect_1,
    src_img_dir=src_img_dir,
    src_mask_dir=PRED_MASK_DIR,
    dest_dir=dest_incorrect,
    src_overlay_dir=source_overlay_dir,
)
